In [42]:
import requests
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from dotenv import load_dotenv
import time

Sites onde tirei as informações

"https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset?resource=download&select=movies_metadata.csv"

"https://datasets.imdbws.com/"

"https://www.themoviedb.org/"

In [43]:
# Leitura dos arquivos csv e tsv

df_movies = pd.read_csv('movies_metadata.csv')

df_basics = pd.read_table('title.basics.tsv.gz')

df_rating = pd.read_table('title.ratings.tsv.gz')

C:\Users\ANALUISASILVADACUNHA\AppData\Local\Temp\ipykernel_18276\3404999718.py:3: DtypeWarning: Columns (0: popularity) have mixed types. Specify dtype option on import or set low_memory=False.
  df_movies = pd.read_csv('movies_metadata.csv')


In [44]:
# Analisamos todas as colunas
df_movies.columns

# Escolhemos apenas as colunas uteis para analise
colunas_uteis = ['id', 'imdb_id', 'title', 'budget', 'revenue', 'release_date', 'runtime', 'popularity', 'vote_average', 'vote_count', 'genres']

# Substituimos o dataset apenas com as colunas uteis
df_movies = df_movies[colunas_uteis]

In [45]:
# Analisamos todas as colunas
df_basics.columns

# Escolhemos apenas as colunas uteis para analise
colunas_uteis = ['tconst', 'titleType', 'primaryTitle', 'startYear', 'runtimeMinutes', 'genres']

# Substituimos o dataset apenas com as colunas uteis
df_basics = df_basics[colunas_uteis]

In [46]:
# Filtrando por apenas filmes porque possui serie e curtas
df_basics = df_basics[df_basics['titleType'] == 'movie']

# Dropando a coluna porque esta filtrado para apenas filmes
df_basics = df_basics.drop('titleType', axis=1)

In [47]:
# Analise dos nulos antes do tratamento
print(df_movies.isnull().sum(), '\n')
print(df_basics.isnull().sum(), '\n')
print(df_rating.isnull().sum(), '\n')

id                0
imdb_id          17
title             6
budget            0
revenue           6
release_date     87
runtime         263
popularity        5
vote_average      6
vote_count        6
genres            0
dtype: int64 

tconst            0
primaryTitle      3
startYear         0
runtimeMinutes    0
genres            0
dtype: int64 

tconst           0
averageRating    0
numVotes         0
dtype: int64 



In [48]:
# Dropando valores nulos em colunas essenciais

df_basics = df_basics.dropna(subset=['primaryTitle'])

df_movies = df_movies.dropna(subset=['imdb_id', 'title', 'release_date', 'runtime'])

In [49]:
# Analise dos nulos após tratamento
print(df_movies.isnull().sum(), '\n')
print(df_basics.isnull().sum(), '\n')
print(df_rating.isnull().sum(), '\n')

id              0
imdb_id         0
title           0
budget          0
revenue         0
release_date    0
runtime         0
popularity      0
vote_average    0
vote_count      0
genres          0
dtype: int64 

tconst            0
primaryTitle      0
startYear         0
runtimeMinutes    0
genres            0
dtype: int64 

tconst           0
averageRating    0
numVotes         0
dtype: int64 



In [50]:
# Renomeando colunas para  merge

df_basics.rename(columns={'tconst': 'imdb_id'}, inplace=True)

df_rating.rename(columns={'tconst': 'imdb_id'}, inplace=True)


In [51]:
# Removemos a coluna starYear porque usaremos a coluna release_date do df_movies

df_basics = df_basics.drop('startYear', axis=1)

# Removemos a coluna runtimeMinutes porque usaremos a coluna runtime de df_movies

df_basics = df_basics.drop('runtimeMinutes', axis=1)

# Removemos a coluna genres de df_movies porque esta em json

df_movies = df_movies.drop('genres', axis=1)

# Removemos as colunas de vote do df_movies por causa que podem estar desatualizadas enquanto as do df_rating são atualizadas pelo imdb

df_movies = df_movies.drop(columns=['vote_average', 'vote_count'])

In [52]:
# Visualizando como ficou o dataframe
df_movies.head()

,id,imdb_id,title,budget,revenue,release_date,runtime,popularity
0,862,tt0114709,Toy Story,30000000,373554033.0,1995-10-30,81.0,21.946943
1,8844,tt0113497,Jumanji,65000000,262797249.0,1995-12-15,104.0,17.015539
2,15602,tt0113228,Grumpier Old Men,0,0.0,1995-12-22,101.0,11.7129
3,31357,tt0114885,Waiting to Exhale,16000000,81452156.0,1995-12-22,127.0,3.859495
4,11862,tt0113041,Father of the Bride Part II,0,76578911.0,1995-02-10,106.0,8.387519


In [53]:
# Visualizando como ficou o dataframe
df_rating.head()

,imdb_id,averageRating,numVotes
0,tt0000001,5.7,2211
1,tt0000002,5.4,319
2,tt0000003,6.4,2344
3,tt0000004,5.1,192
4,tt0000005,6.2,3059


In [54]:
# Visualizando como ficou o dataframe
df_basics.head()

,imdb_id,primaryTitle,genres
8,tt0000009,Miss Jerry,Romance
144,tt0000147,The Corbett-Fitzsimmons Fight,"Documentary,News,Sport"
498,tt0000502,Bohemios,\N
570,tt0000574,The Story of the Kelly Gang,"Action,Adventure,Biography"
587,tt0000591,The Prodigal Son,Drama


In [55]:
# Começando o merge
# Primeiro merge entre rating e movies
df_final = pd.merge(df_movies, df_rating, on='imdb_id', how='inner')

# Segundo merge que junto o basics apenas com as colunas imdb e genres
df_final = pd.merge(df_final, df_basics[['imdb_id', 'genres']], on='imdb_id', how='inner')

In [56]:
# Verificando o dataframe combinado
df_final.head()

,id,imdb_id,title,budget,revenue,release_date,runtime,popularity,averageRating,numVotes,genres
0,862,tt0114709,Toy Story,30000000,373554033.0,1995-10-30,81.0,21.946943,8.3,1176535,"Adventure,Animation,Comedy"
1,8844,tt0113497,Jumanji,65000000,262797249.0,1995-12-15,104.0,17.015539,7.1,414470,"Adventure,Comedy,Family"
2,15602,tt0113228,Grumpier Old Men,0,0.0,1995-12-22,101.0,11.7129,6.7,31720,"Comedy,Romance"
3,31357,tt0114885,Waiting to Exhale,16000000,81452156.0,1995-12-22,127.0,3.859495,6.0,14008,"Comedy,Drama,Romance"
4,11862,tt0113041,Father of the Bride Part II,0,76578911.0,1995-02-10,106.0,8.387519,6.1,45316,"Comedy,Family,Romance"


In [57]:
# Verificando orçamentos e faturamentos zerados
# Convertemos primeiro para float caso não esteja
df_final['budget'] = df_final['budget'].astype(float)
df_final['revenue'] = df_final['revenue'].astype(float)

qtd_zeros_budget = (df_final['budget'] == 0).sum()
qtd_zeros_revenue = (df_final['revenue'] == 0).sum()

print(f"Orçamentos zerados: {qtd_zeros_budget}")
print(f"Faturamentos zerados: {qtd_zeros_revenue}")

# Verificando as strings nulas do IMDb (\N)
qtd_zeros_genres = (df_final['genres'] == '\\N').sum()

print(f"Gêneros como '\\N': {qtd_zeros_genres}")

Orçamentos zerados: 30423
Faturamentos zerados: 31513
Gêneros como '\N': 12


In [58]:
# Escolhemos tirar todos os filmes que não possuem orçamento e faturamento
df_final = df_final[(df_final['budget'] > 0) & (df_final['revenue'] > 0)]

# Limpando os falsos nulos do IMDb
df_final = df_final[df_final['genres'] != '\\N']

# Mexemos também no runtime para não ter zeros
df_final['runtime'] = df_final['runtime'].astype(float)
df_final = df_final[df_final['runtime'] > 0]

In [59]:
# Dataset após organização
df_final.head()

,id,imdb_id,title,budget,revenue,release_date,runtime,popularity,averageRating,numVotes,genres
0,862,tt0114709,Toy Story,30000000.0,373554033.0,1995-10-30,81.0,21.946943,8.3,1176535,"Adventure,Animation,Comedy"
1,8844,tt0113497,Jumanji,65000000.0,262797249.0,1995-12-15,104.0,17.015539,7.1,414470,"Adventure,Comedy,Family"
3,31357,tt0114885,Waiting to Exhale,16000000.0,81452156.0,1995-12-22,127.0,3.859495,6.0,14008,"Comedy,Drama,Romance"
5,949,tt0113277,Heat,60000000.0,187436818.0,1995-12-15,170.0,17.924927,8.3,806505,"Action,Crime,Drama"
8,9091,tt0114576,Sudden Death,35000000.0,64350171.0,1995-12-22,106.0,5.23158,5.9,39661,"Action,Crime,Thriller"


In [60]:
# Vendo as informações do dataset
df_final.info()

<class 'pandas.DataFrame'>
Index: 5355 entries, 0 to 38846
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             5355 non-null   str    
 1   imdb_id        5355 non-null   str    
 2   title          5355 non-null   str    
 3   budget         5355 non-null   float64
 4   revenue        5355 non-null   float64
 5   release_date   5355 non-null   str    
 6   runtime        5355 non-null   float64
 7   popularity     5355 non-null   object 
 8   averageRating  5355 non-null   float64
 9   numVotes       5355 non-null   int64  
 10  genres         5355 non-null   str    
dtypes: float64(4), int64(1), object(1), str(5)
memory usage: 802.9+ KB


In [61]:
# Convertendo release_date em date e popularity em float

df_final['release_date'] = pd.to_datetime(df_final['release_date'], format='%Y-%m-%d')

df_final['popularity'] = df_final['popularity'].astype(float)

In [62]:
# Vendo se os tipos estão agora certos
df_final.info()

<class 'pandas.DataFrame'>
Index: 5355 entries, 0 to 38846
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id             5355 non-null   str           
 1   imdb_id        5355 non-null   str           
 2   title          5355 non-null   str           
 3   budget         5355 non-null   float64       
 4   revenue        5355 non-null   float64       
 5   release_date   5355 non-null   datetime64[us]
 6   runtime        5355 non-null   float64       
 7   popularity     5355 non-null   float64       
 8   averageRating  5355 non-null   float64       
 9   numVotes       5355 non-null   int64         
 10  genres         5355 non-null   str           
dtypes: datetime64[us](1), float64(5), int64(1), str(4)
memory usage: 749.9 KB


In [63]:
# Configurando a API do TMDB

load_dotenv("tmdb.env")
API_KEY = os.getenv("TMDB_API_KEY")

if not API_KEY:
    raise ValueError("TMDB_API_KEY não encontrada no .env. Verifique o arquivo.")

BASE_URL = "https://api.themoviedb.org/3"

In [64]:
# Função para buscar detalhes de um filme pelo id do TMDB
# Com timeout e retry: sem isso, uma única requisição que trava
# pode deixar o loop inteiro parado para sempre

def buscar_detalhes_filme(tmdb_id, max_tentativas=2, timeout=8):
    url = f"{BASE_URL}/movie/{int(tmdb_id)}"
    params = {"api_key": API_KEY, "language": "en-US"}

    for tentativa in range(max_tentativas):
        try:
            return requests.get(url, params=params, timeout=timeout)
        except requests.exceptions.RequestException:
            if tentativa < max_tentativas - 1:
                time.sleep(1)
                continue
            return None
    return None

In [65]:
# Extração principal: para cada filme, busca franquia, estúdio, idioma e país
# Salva em cache (tmdbAPI_dados.csv) para não precisar chamar a API de novo
# toda vez que o notebook for reaberto

if os.path.exists("tmdbAPI_dados.csv"):
    df_api = pd.read_csv("tmdbAPI_dados.csv")
    print("Carregado do cache, sem chamar a API.")

else:
    resultados = []
    total = len(df_final)
    falhas = 0
    inicio = time.time()

    for i, row in df_final.iterrows():
        resp = buscar_detalhes_filme(row["id"])

        if resp is not None and resp.status_code == 200:
            dados = resp.json()

            colecao = dados.get("belongs_to_collection")
            franquia = colecao["name"] if colecao else None

            estudios = dados.get("production_companies", [])
            estudio_principal = estudios[0]["name"] if estudios else None

            paises = dados.get("production_countries", [])
            pais_producao = ", ".join([p["name"] for p in paises]) if paises else None

            resultados.append({
                "id": row["id"],
                "franquia": franquia,
                "is_franchise": colecao is not None,
                "estudio_principal": estudio_principal,
                "original_language": dados.get("original_language"),
                "production_countries": pais_producao,
                "status": dados.get("status"),
            })

        else:
            falhas += 1
            resultados.append({
                "id": row["id"],
                "franquia": None,
                "is_franchise": None,
                "estudio_principal": None,
                "original_language": None,
                "production_countries": None,
                "status": None,
            })

        if (i + 1) % 10 == 0 or (i + 1) == total:
            decorrido = time.time() - inicio
            print(f"Progresso: {i + 1}/{total} | falhas: {falhas} | tempo: {decorrido:.0f}s")

        time.sleep(0.03)

    df_api = pd.DataFrame(resultados)
    df_api.to_csv("tmdbAPI_dados.csv", index=False)
    print(f"\nConcluído e salvo em cache. {falhas} filme(s) sem dados.")

Carregado do cache, sem chamar a API.


In [66]:
# Visualizando como ficou o dataframe da API
df_api.head()

,id,franquia,is_franchise,estudio_principal,original_language,production_countries,status
0,862,Toy Story Collection,True,Pixar,en,United States of America,Released
1,8844,Jumanji Collection,True,TriStar Pictures,en,United States of America,Released
2,31357,NaN,False,20th Century Fox,en,United States of America,Released
3,949,Heat Collection,True,Warner Bros. Pictures,en,United States of America,Released
4,9091,NaN,False,Shattered Productions,en,United States of America,Released


In [67]:
# Todos os filmes são released então podemos dropar a coluna status
df_api['status'].value_counts()

df_api = df_api.drop(columns=['status'])

In [68]:
df_api.isnull().sum()

id                         0
franquia                3970
is_franchise               0
estudio_principal         45
original_language          0
production_countries      10
dtype: int64

In [69]:
# Renomeando  para não ficar nulo
df_api['franquia'] = df_api['franquia'].fillna('Sem franquia')

# Dropando a coluna is_franchise porque ao nomear os nulos da coluna franquia já é possível analisar
df_api = df_api.drop(columns=['is_franchise'])

# Renomeando quem não tem estudio principal
df_api['estudio_principal'] = df_api['estudio_principal'].fillna('Não informado')

# Nomeando quem não tem paises de produção
df_api['production_countries'] = df_api['production_countries'].fillna('Não informado')

# Verificando se tratou todos os nulos
df_api.isnull().sum()

id                      0
franquia                0
estudio_principal       0
original_language       0
production_countries    0
dtype: int64

In [70]:
# Convertendo para string para o merge
df_api['id'] = df_api['id'].astype(str)

# Merge final usando o id do TMDB como chave
df_final = pd.merge(df_final, df_api, on="id", how="left")

In [ ]:
# Visualizando como ficou o dataframe merge final
df_final.head()

,id,imdb_id,title,budget,revenue,release_date,runtime,popularity,averageRating,numVotes,genres,franquia,estudio_principal,original_language,production_countries
0,862,tt0114709,Toy Story,30000000.0,373554033.0,1995-10-30,81.0,21.946943,8.3,1176535,"Adventure,Animation,Comedy",Toy Story Collection,Pixar,en,United States of America
1,8844,tt0113497,Jumanji,65000000.0,262797249.0,1995-12-15,104.0,17.015539,7.1,414470,"Adventure,Comedy,Family",Jumanji Collection,TriStar Pictures,en,United States of America
2,31357,tt0114885,Waiting to Exhale,16000000.0,81452156.0,1995-12-22,127.0,3.859495,6.0,14008,"Comedy,Drama,Romance",Sem franquia,20th Century Fox,en,United States of America
3,949,tt0113277,Heat,60000000.0,187436818.0,1995-12-15,170.0,17.924927,8.3,806505,"Action,Crime,Drama",Heat Collection,Warner Bros. Pictures,en,United States of America
4,9091,tt0114576,Sudden Death,35000000.0,64350171.0,1995-12-22,106.0,5.231580,5.9,39661,"Action,Crime,Thriller",Sem franquia,Shattered Productions,en,United States of America


In [75]:
# Renomeando as colunas para melhor orientação e clareza
df_final = df_final.rename(columns={
    'id':                   'tmdb_id',
    'budget':               'orcamento',
    'revenue':              'receita',
    'release_date':         'data_lancamento',
    'runtime':              'duracao_min',
    'popularity':           'popularidade',
    'averageRating':        'nota_media',
    'numVotes':             'num_votos',
    'genres':               'generos',
    'estudio_principal':    'estudio',
    'original_language':    'idioma_original',
    'production_countries': 'paises_producao',
})

In [ ]:
# Vendo as informações do dataframe final
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 5367 entries, 0 to 5366
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   tmdb_id          5367 non-null   str           
 1   imdb_id          5367 non-null   str           
 2   title            5367 non-null   str           
 3   orcamento        5367 non-null   float64       
 4   receita          5367 non-null   float64       
 5   data_lancamento  5367 non-null   datetime64[us]
 6   duracao_min      5367 non-null   float64       
 7   popularidade     5367 non-null   float64       
 8   nota_media       5367 non-null   float64       
 9   num_votos        5367 non-null   int64         
 10  generos          5367 non-null   str           
 11  franquia         5367 non-null   str           
 12  estudio          5367 non-null   str           
 13  idioma_original  5367 non-null   str           
 14  paises_producao  5367 non-null   str           
dty

In [77]:
# Criando coluna de lucro e ROI para análises de performance financeira
df_final['lucro'] = df_final['receita'] - df_final['orcamento']
df_final['roi'] = (df_final['lucro'] / df_final['orcamento']) * 100

# Extraindo ano e agrupando por década para análise de período
df_final['ano'] = df_final['data_lancamento'].dt.year
df_final['decada'] = (df_final['ano'] // 10 * 10).astype(str) + 's'

# Simplificando idioma e país para comparação binária
# Analise para responder se a lingua inglesa ou ser feito no EUA influencia as perfomances dos filmes
df_final['idioma_ingles'] = df_final['idioma_original'] == 'en'
df_final['produzido_eua'] = df_final['paises_producao'].str.contains('United States of America', na=False)

# Extraindo o primeiro gênero listado como gênero principal
df_final['genero_principal'] = df_final['generos'].str.split(',').str[0]

# Filmes acima da mediana de votos têm nota mais representativa
# Para analise de correlação entre numero de votos e nota média 
mediana_votos = df_final['num_votos'].median()
df_final['voto_confiavel'] = df_final['num_votos'] >= mediana_votos

In [78]:
# Verificando o dataframe após adição das colunas novas
df_final.head()

,tmdb_id,imdb_id,title,orcamento,receita,data_lancamento,duracao_min,popularidade,nota_media,num_votos,...,idioma_original,paises_producao,lucro,roi,ano,decada,idioma_ingles,produzido_eua,genero_principal,voto_confiavel
0,862,tt0114709,Toy Story,30000000.0,373554033.0,1995-10-30,81.0,21.946943,8.3,1176535,...,en,United States of America,343554033.0,1145.180110,1995,1990s,True,True,Adventure,True
1,8844,tt0113497,Jumanji,65000000.0,262797249.0,1995-12-15,104.0,17.015539,7.1,414470,...,en,United States of America,197797249.0,304.303460,1995,1990s,True,True,Adventure,True
2,31357,tt0114885,Waiting to Exhale,16000000.0,81452156.0,1995-12-22,127.0,3.859495,6.0,14008,...,en,United States of America,65452156.0,409.075975,1995,1990s,True,True,Comedy,False
3,949,tt0113277,Heat,60000000.0,187436818.0,1995-12-15,170.0,17.924927,8.3,806505,...,en,United States of America,127436818.0,212.394697,1995,1990s,True,True,Action,True
4,9091,tt0114576,Sudden Death,35000000.0,64350171.0,1995-12-22,106.0,5.231580,5.9,39661,...,en,United States of America,29350171.0,83.857631,1995,1990s,True,True,Action,False


In [88]:
# Checando NaN nativo do pandas
print(df_final['roi'].isna().sum())

# Checando os valores únicos suspeitos
print(df_final[df_final['roi'] == 0][['title', 'orcamento', 'receita', 'lucro', 'roi']])

0
Empty DataFrame
Columns: [title, orcamento, receita, lucro, roi]
Index: []


In [ ]:
# Removendo filmes sem dados financeiros reais (orcamento == receita, valores sem utilidade analitica)
df_final = df_final[df_final['lucro'] != 0].reset_index(drop=True)

# Verificando as informações
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 5348 entries, 0 to 5347
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   tmdb_id           5348 non-null   str           
 1   imdb_id           5348 non-null   str           
 2   title             5348 non-null   str           
 3   orcamento         5348 non-null   float64       
 4   receita           5348 non-null   float64       
 5   data_lancamento   5348 non-null   datetime64[us]
 6   duracao_min       5348 non-null   float64       
 7   popularidade      5348 non-null   float64       
 8   nota_media        5348 non-null   float64       
 9   num_votos         5348 non-null   int64         
 10  generos           5348 non-null   str           
 11  franquia          5348 non-null   str           
 12  estudio           5348 non-null   str           
 13  idioma_original   5348 non-null   str           
 14  paises_producao   5348 non-null   s

In [ ]:
# Filmes com orçamento abaixo de 1000 são claramente suspeitos então removendo
# Descobri após analise de roi muito altos justificados por causa de orcamentos suspeitos
df_final = df_final[df_final['orcamento'] >= 1000].reset_index(drop=True)

In [93]:
# Exportando o dataframe final limpo
df_final.to_csv('df_final_limpo.csv', index=False)
print(df_final.shape)

(5299, 23)
